### Bronz Layer
Incremental bronz layer ingestion with rerun-safe watermark logic.

### Step 1- Imports nd setup

This cess imports pyspark and delta helpers used in the notebook, switches to the correct catalog and makes sure the bronze schema exists before we start loading data.



In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("USE CATALOG `ecom_azure_project`")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS `ecom_azure_project`.bronze_schema")


### Step 2

This table stores the watermark fro each source table.

it helps the pipeline remember:
- the latest timestamp already processed
- the latest primary key processed at that timestamp
- how many rows were written in the latest run

In [0]:
spark.sql("""
          create table if not exists ecom_azure_project.bronze_schema.ingestion_control(
              layer string,
              table_name string,
              ts_col string,
              pk_col string,
              last_successfull_ts timestamp,
              last_successfull_pk bigint,
              last_run_id string,
              rows_written bigint, 
              run_status string,
              updated_at timestamp            
            )
            using delta
          
          
          
          """)

### Step 3 - Source table configuration

this cess defines which source tabled will be oaded into bronze and which columns should be used as:
- primary key
- timestamp/watermark column

it also creates a unique bronze_run_id fro the current pipeline run.

In [0]:
tables_config = {

    "orders": { "pk_col" : "order_id", "ts_col" : "updated_at"},
    "products" : {"pk_col" : "product_id", "ts_col" : "updated_at"},
    "payments" : {"pk_col" : "payment_id", "ts_col" : "processed_at"}
}

bronze_run_id = str(uuid.uuid4())
print("current bronze run id:" , bronze_run_id)

### Step 4  helper functions

this cess contains reusable functions:
- get_last_successfull_watermark() reads the last processed waermark from the control table
- upsert_bronze_control() updatesthe control tableafter a successfullbronz load

these functions keep the main logic cleaner and easier to understand.


In [0]:
def get_last_successful_watermark(table_name: str):
    ctrl = (
        spark.table("ecom_azure_project.bronze_schema.ingestion_control")
        .filter(
            (f.col("layer") == "bronze")&
            (f.col("table_name") == table_name)&
            (f.col("run_status")=="success")
        )
        .orderBy(f.col("updated_at").desc())
        .limit(1)
    )
    rows = ctrl.collect()

    if not rows:
        return None, None
    
    return rows[0]["last_successfull_ts"], rows[0]["last_successfull_pk"]

In [0]:
def upsert_bronze_control(table_name, ts_col, pk_col, last_ts, last_pk, rows_written, run_id):
    control_df = spark.createDataFrame(
        [
            (
                "bronze",
                table_name,
                ts_col,
                pk_col,
                last_ts,
                int(last_pk) if last_pk is not None else None,
                run_id,
                int(rows_written),
                "success",
                datetime.utcnow()
            )
        ],
        schema="""
        layer string,
        table_name string,
        ts_col string,
        pk_col string,
        last_successfull_ts timestamp,
        last_successfull_pk bigint,
        last_run_id string,
        rows_written bigint,
        run_status string,
        updated_at timestamp
        
        
        """   
        )
    dt = DeltaTable.forName(spark, "ecom_azure_project.bronze_schema.ingestion_control")
    dt.alias("t").merge(
        control_df.alias("s"),
        "t.table_name = s.table_name and t.layer = s.layer").whenMatchedUpdate(set={
            "ts_col" : "s.ts_col",
            "pk_col" : "s.pk_col",
            "last_successfull_ts" : "s.last_successfull_ts",
            "last_successfull_pk" : "s.last_successfull_pk",
            "last_run_id" : "s.last_run_id",
            "rows_written" : "s.rows_written",
            "run_status" : "s.run_status",
            "updated_at" : "s.updated_at"
        }).whenNotMatchedInsertAll().execute()
    





### Step 5 - Broze incremental load loop

This is the main bronze ayer logic.

For each table, the notebook:

- reads the last watermark
- reads the source SQL table
- filters only new/changed rows
- adds bronze audit columns
- appends the rows into the bronze delta table
- updates the control table

This is the core incremental logic.

In [0]:
f = F

for table_name, cfg in tables_config.items():
    ts_col = cfg["ts_col"]
    pk_col = cfg["pk_col"]
    source_table = f"`ecom-db-to-databricks_catalog`.dbo.{table_name}"
    target_table = f"ecom_azure_project.bronze_schema.{table_name}_raw"
    last_successful_ts, last_successful_pk = get_last_successful_watermark(table_name)
    print(f"\n=== Processing {table_name} ===")
    print(f"Last successful ts: {last_successful_ts}")
    print(f"Last successful pk: {last_successful_pk}")
    
    source_df = spark.read.table(source_table).withColumn(ts_col, F.col(ts_col).cast("timestamp"))
    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (F.col(ts_col) > F.lit(last_successful_ts)) |
        (
           (F.col(ts_col) == F.lit(last_successful_ts)) &
           (F.col(pk_col).cast("long") > F.lit(int(last_successful_pk)))
        )
        )
        
    rows_to_load = (
        rows_to_load
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("bronze_run_id", F.lit(bronze_run_id))
        .withColumn("bronze_source_table", F.lit(source_table))
    )
    row_count = rows_to_load.count()
    print(f"{table_name} rows_to_load = {row_count}")
    
    if row_count == 0:
        print(f"No new rows to load for {table_name}.")
        upsert_bronze_control(
        table_name,
        ts_col,
        pk_col,
        last_successful_ts,
        last_successful_pk,
        row_count,
        bronze_run_id
    )


    rows_to_load.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(target_table)

    max_ts = rows_to_load.agg(F.max(ts_col).alias("max_ts")).collect()[0]["max_ts"]
    max_pk = (
        rows_to_load
        .filter(F.col(ts_col) == F.lit(max_ts))
        .agg(F.max(F.col(pk_col).cast("long")).alias("max_pk"))
        .collect()[0]["max_pk"]
        )

    upsert_bronze_control(table_name,ts_col, pk_col,max_ts,max_pk,row_count,bronze_run_id)            
    print(f"Wrote {row_count} rows to {target_table}")

 

In [0]:
orders_table = "ecom_azure_project.bronze_schema.orders_raw"
products_table = "ecom_azure_project.bronze_schema.products_raw"
payments_table = "ecom_azure_project.bronze_schema.payments_raw"

if spark.catalog.tableExists(orders_table):
    print("Orders Bronze Count :", spark.sql(f"select count(*) from {orders_table}").collect()[0][0])
else:
    print("Orders Bronze Count : table not found")

if spark.catalog.tableExists(products_table):
    print("Products Bronze Count :", spark.sql(f"select count(*) from {products_table}").collect()[0][0])
else:
    print("Products Bronze Count : table not found")

if spark.catalog.tableExists(payments_table):
    print("Payments Bronze Count :", spark.sql(f"select count(*) from {payments_table}").collect()[0][0])
else:
    print("Payments Bronze Count : table not found")

display(spark.sql("select * from ecom_azure_project.bronze_schema.ingestion_control").orderBy("table_name"))